In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ============================================================
# CONFIG
# ============================================================
DATA_DIR = "/kaggle/input/recodai-luc-scientific-image-forgery-detection"
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train_images")
TRAIN_MASK_DIR = os.path.join(DATA_DIR, "train_masks")
TEST_IMG_DIR = os.path.join(DATA_DIR, "test_images")

IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 2
LR = 1e-3
THRESHOLD = 0.3

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# RLE
# ============================================================
def rle_encode(mask):
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return list(runs)

# ============================================================
# DATASET
# ============================================================
class ForgeryDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = []

        for root, _, files in os.walk(img_dir):
            for f in files:
                if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff")):
                    self.images.append(os.path.join(root, f))

        print(f"Loaded {len(self.images)} training images")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        name = os.path.basename(img_path)

        image = cv2.imread(img_path)
        if image is None:
            image = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

        image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
        image = image.transpose(2, 0, 1) / 255.0

        mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)
        base = os.path.splitext(name)[0]

        for f in os.listdir(self.mask_dir):
            if f.startswith(base):
                m = cv2.imread(os.path.join(self.mask_dir, f), 0)
                if m is not None:
                    m = cv2.resize(m, (IMG_SIZE, IMG_SIZE))
                    mask = np.maximum(mask, m > 0)

        return (
            torch.tensor(image, dtype=torch.float32),
            torch.tensor(mask[None], dtype=torch.float32),
        )

# ============================================================
# TINY U-NET
# ============================================================
class UNetSmall(nn.Module):
    def __init__(self):
        super().__init__()

        def C(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.ReLU(inplace=True),
            )

        self.d1 = C(3, 16)
        self.p1 = nn.MaxPool2d(2)
        self.d2 = C(16, 32)
        self.p2 = nn.MaxPool2d(2)
        self.mid = C(32, 64)

        self.u2 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.c2 = C(64, 32)
        self.u1 = nn.ConvTranspose2d(32, 16, 2, 2)
        self.c1 = C(32, 16)
        self.out = nn.Conv2d(16, 1, 1)

    def forward(self, x):
        d1 = self.d1(x)
        d2 = self.d2(self.p1(d1))
        m = self.mid(self.p2(d2))
        x = self.c2(torch.cat([self.u2(m), d2], 1))
        x = self.c1(torch.cat([self.u1(x), d1], 1))
        return self.out(x)

# ============================================================
# TRAIN
# ============================================================
train_ds = ForgeryDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

model = UNetSmall().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCEWithLogitsLoss()

model.train()
for imgs, masks in tqdm(train_loader):
    imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
    optimizer.zero_grad()
    loss = criterion(model(imgs), masks)
    loss.backward()
    optimizer.step()

# ============================================================
# INFERENCE
# ============================================================
model.eval()
results = []

for fname in tqdm(sorted(os.listdir(TEST_IMG_DIR))):
    if not fname.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff")):
        continue

    case_id = int(os.path.splitext(fname)[0])
    img = cv2.imread(os.path.join(TEST_IMG_DIR, fname))

    if img is None:
        results.append((case_id, "authentic"))
        continue

    img_r = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    x = torch.tensor(img_r.transpose(2, 0, 1) / 255.0).float().unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred = torch.sigmoid(model(x))[0, 0].cpu().numpy()

    mask = (pred > THRESHOLD).astype(np.uint8)
    mask = cv2.resize(mask, (img.shape[1], img.shape[0]))

    if mask.sum() < 30:
        results.append((case_id, "authentic"))
    else:
        results.append((case_id, str(rle_encode(mask))))

# ============================================================
# SUBMISSION
# ============================================================
pd.DataFrame(results, columns=["case_id", "annotation"]).to_csv(
    "submission.csv", index=False
)

print("✅ submission.csv ready")

Loaded 5128 training images


100%|██████████| 1/1 [00:00<00:00, 11.57it/s]

✅ submission.csv ready
